# 04 Results Visualization

Publication-ready figures for AED-XAI (BMVC 2026).

All figures are saved to `../figures/` as PDF + PNG at 300 DPI.

In [ ]:
import sys
sys.path.insert(0, "..")

import base64
import json
import warnings
import zlib
from pathlib import Path

import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
from IPython.display import display

RESULTS_DIR = Path("../data/results")
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-paper")
matplotlib.rcParams.update({"font.size": 11, "font.family": "serif"})

METHODS = ["GradCAM", "G-CAME", "D-CLOSE", "LIME", "AED-XAI (Ours)"]
METHOD_COLORS = {
    "GradCAM": "#4878CF",
    "G-CAME": "#6ACC65",
    "D-CLOSE": "#D65F5F",
    "LIME": "#B47CC7",
    "AED-XAI (Ours)": "#C4AD66",
}

def save_figure(fig, name):
    fig.savefig(FIGURES_DIR / f"{name}.pdf", dpi=300, bbox_inches="tight")
    fig.savefig(FIGURES_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    print(f"Saved {name}.pdf + .png")

print(f"Results directory : {RESULTS_DIR.resolve()}")
print(f"Figures directory : {FIGURES_DIR.resolve()}")

## 1. Load Experiment Results

In [ ]:
def load_or_synth(filename, synth_fn):
    path = RESULTS_DIR / filename
    if path.exists():
        print(f"Loaded {path}")
        return pd.read_csv(path)
    warnings.warn(f"{filename} not found — using synthetic placeholder data")
    return synth_fn()

def clip01(values):
    return np.clip(values, 0.0, 1.0)

def synth_main_results(seed=42, n=500):
    rng = np.random.default_rng(seed)
    profiles = {
        "GradCAM": {"composite": (0.66, 0.09), "pg": (0.73, 0.12), "ebpg": (0.55, 0.12), "oa": (0.55, 0.13), "sparsity": (0.61, 0.10), "time": (0.45, 0.08)},
        "G-CAME": {"composite": (0.64, 0.09), "pg": (0.76, 0.11), "ebpg": (0.60, 0.11), "oa": (0.50, 0.13), "sparsity": (0.66, 0.09), "time": (0.55, 0.10)},
        "D-CLOSE": {"composite": (0.62, 0.10), "pg": (0.69, 0.13), "ebpg": (0.63, 0.12), "oa": (0.58, 0.14), "sparsity": (0.55, 0.11), "time": (28.0, 5.0)},
        "LIME": {"composite": (0.55, 0.10), "pg": (0.62, 0.14), "ebpg": (0.50, 0.12), "oa": (0.42, 0.15), "sparsity": (0.58, 0.12), "time": (12.0, 2.5)},
        "AED-XAI (Ours)": {"composite": (0.72, 0.08), "pg": (0.79, 0.10), "ebpg": (0.66, 0.10), "oa": (0.65, 0.12), "sparsity": (0.70, 0.09), "time": (3.8, 1.2)},
    }
    rows = []
    complexities = np.array(["low", "medium", "high"])
    complexity_probs = np.array([0.35, 0.45, 0.20])
    for method in METHODS:
        profile = profiles[method]
        for image_id in range(n):
            scene_complexity = str(rng.choice(complexities, p=complexity_probs))
            det_base = {"low": 3, "medium": 8, "high": 15}[scene_complexity]
            rows.append({
                "image_id": image_id,
                "method": method,
                "pg": float(clip01(rng.normal(*profile["pg"]))),
                "ebpg": float(clip01(rng.normal(*profile["ebpg"]))),
                "oa": float(np.clip(rng.normal(*profile["oa"]), -0.5, 1.0)),
                "sparsity": float(clip01(rng.normal(*profile["sparsity"]))),
                "composite": float(clip01(rng.normal(*profile["composite"]))),
                "computation_time": float(max(0.02, rng.normal(*profile["time"]))),
                "scene_complexity": scene_complexity,
                "num_detections": int(max(1, rng.poisson(det_base))),
            })
    return pd.DataFrame(rows)

def synth_threshold_ablation(seed=42, n=500):
    rng = np.random.default_rng(seed + 1)
    profiles = {
        "fixed_0.3": (0.655, 0.085, 0.9, 0.84),
        "fixed_0.5": (0.685, 0.080, 1.4, 0.88),
        "fixed_0.7": (0.640, 0.095, 2.2, 0.72),
        "percentile_40": (0.728, 0.070, 1.2, 0.93),
        "learned": (0.735, 0.068, 1.1, 0.95),
    }
    rows = []
    for mode, (mean, std, refine_mean, conv_prob) in profiles.items():
        for image_id in range(n):
            rows.append({
                "threshold_mode": mode,
                "image_id": image_id,
                "composite": float(clip01(rng.normal(mean, std))),
                "refine_calls": int(np.clip(rng.poisson(refine_mean), 0, 4)),
                "converged": bool(rng.random() < conv_prob),
            })
    return pd.DataFrame(rows)

def synth_selector_ablation(seed=42, n=500):
    rng = np.random.default_rng(seed + 2)
    oracle_methods = rng.choice(["GradCAM", "G-CAME", "D-CLOSE", "LIME"], size=n, p=[0.25, 0.35, 0.25, 0.15])
    profiles = {
        "always_gradcam": (0.25, 0.61, 0.09),
        "rule_based": (0.55, 0.68, 0.08),
        "oracle_mlp": (0.78, 0.72, 0.07),
    }
    rows = []
    for selector_type, (accuracy, comp_mean, comp_std) in profiles.items():
        for image_id, oracle in enumerate(oracle_methods):
            correct = rng.random() < accuracy
            if selector_type == "always_gradcam":
                chosen = "GradCAM"
                correct = chosen == oracle
            elif correct:
                chosen = oracle
            else:
                choices = [m for m in ["GradCAM", "G-CAME", "D-CLOSE", "LIME"] if m != oracle]
                chosen = str(rng.choice(choices))
            rows.append({
                "selector_type": selector_type,
                "image_id": image_id,
                "method_chosen": chosen,
                "oracle_method": oracle,
                "correct": bool(chosen == oracle),
                "composite": float(clip01(rng.normal(comp_mean + (0.025 if chosen == oracle else -0.025), comp_std))),
            })
    return pd.DataFrame(rows)

def synth_cross_domain(seed=42, n=200):
    rng = np.random.default_rng(seed + 3)
    domain_drop = {"COCO": 0.00, "VOC": 0.04, "VisDrone": 0.11}
    base = {"GradCAM": 0.66, "G-CAME": 0.64, "D-CLOSE": 0.62, "LIME": 0.55, "AED-XAI (Ours)": 0.72}
    rows = []
    for domain, drop in domain_drop.items():
        for method in METHODS:
            for _ in range(n):
                composite = clip01(rng.normal(base[method] - drop, 0.075))
                rows.append({
                    "domain": domain,
                    "method": method,
                    "pg": float(clip01(rng.normal(composite + 0.06, 0.08))),
                    "oa": float(np.clip(rng.normal(composite - 0.06, 0.10), -0.5, 1.0)),
                    "composite": float(composite),
                })
    return pd.DataFrame(rows)

main_results = load_or_synth("main_results.csv", synth_main_results)
threshold_ablation = load_or_synth("threshold_ablation.csv", synth_threshold_ablation)
selector_ablation = load_or_synth("selector_ablation.csv", synth_selector_ablation)
cross_domain = load_or_synth("cross_domain.csv", synth_cross_domain)

for name, df in [
    ("main_results", main_results),
    ("threshold_ablation", threshold_ablation),
    ("selector_ablation", selector_ablation),
    ("cross_domain", cross_domain),
]:
    print(f"\n{name}: shape={df.shape}")
    display(df.head().round(3))

## 2. Table 1: Main Quantitative Comparison (BMVC Table 1)

In [ ]:
metric_cols = ["pg", "ebpg", "oa", "sparsity", "composite", "computation_time"]
summary_mean = main_results.groupby("method")[metric_cols].mean().reindex(METHODS)
summary_std = main_results.groupby("method")[metric_cols].std().reindex(METHODS)

display(summary_mean.round(3))

best_methods = {}
for metric in metric_cols:
    if metric == "computation_time":
        best_methods[metric] = summary_mean[metric].idxmin()
    else:
        best_methods[metric] = summary_mean[metric].idxmax()

formatted_rows = []
for method in METHODS:
    row = {"method": method}
    for metric in metric_cols:
        value = summary_mean.loc[method, metric]
        std = summary_std.loc[method, metric]
        cell = f"{value:.3f} ± {std:.3f}"
        if best_methods[metric] == method:
            cell = f"\\textbf{{{cell}}}"
        row[metric] = cell
    formatted_rows.append(row)

table_df = pd.DataFrame(formatted_rows).set_index("method")
display(table_df)

print("LaTeX source:")
print(table_df.to_latex(float_format="%.3f", bold_rows=False, escape=False))

## 3. Figure 1: Composite Score Distribution by Method

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
data = [main_results[main_results["method"] == method]["composite"].values for method in METHODS]
parts = ax.violinplot(data, showmeans=False, showmedians=True, showextrema=False)
for body, method in zip(parts["bodies"], METHODS):
    body.set_facecolor(METHOD_COLORS[method])
    body.set_edgecolor("black")
    body.set_alpha(0.65)
if "cmedians" in parts:
    parts["cmedians"].set_color("black")

rng = np.random.default_rng(42)
for idx, method in enumerate(METHODS, start=1):
    y = main_results[main_results["method"] == method]["composite"].values
    x = rng.normal(idx, 0.045, size=len(y))
    ax.scatter(x, y, s=3, alpha=0.15, color="black", linewidths=0)

ours_mean = main_results[main_results["method"] == "AED-XAI (Ours)"]["composite"].mean()
ax.axhline(ours_mean, color=METHOD_COLORS["AED-XAI (Ours)"], linestyle="--", linewidth=1.5, label="Ours mean")
ax.set_xticks(np.arange(1, len(METHODS) + 1))
ax.set_xticklabels(METHODS, rotation=15, ha="right")
ax.set_ylabel("Composite Score")
ax.set_xlabel("Method")
ax.set_title("Composite Score Distribution (500 COCO val2017 images)")
ax.set_ylim(0, 1)
ax.legend(frameon=False)
plt.tight_layout()
save_figure(fig, "fig1_composite_distribution")
plt.show()

## 4. Figure 2: Threshold Ablation (Table 4)

In [ ]:
threshold_order = ["fixed_0.3", "fixed_0.5", "fixed_0.7", "percentile_40", "learned"]
thr_group = threshold_ablation.groupby("threshold_mode").agg(
    composite_mean=("composite", "mean"),
    composite_std=("composite", "std"),
    refine_calls_mean=("refine_calls", "mean"),
    converged_pct=("converged", "mean"),
).reindex(threshold_order)

fig, ax1 = plt.subplots(1, 1, figsize=(9, 5))
x = np.arange(len(threshold_order))
bars = ax1.bar(x, thr_group["composite_mean"], yerr=thr_group["composite_std"], capsize=4, color="#6A8DDB", edgecolor="black")
for bar, mode in zip(bars, threshold_order):
    if mode in {"percentile_40", "learned"}:
        bar.set_hatch("///")

best_idx = int(np.nanargmax(thr_group["composite_mean"].values))
ax1.text(best_idx, thr_group["composite_mean"].iloc[best_idx] + 0.035, "Best", ha="center", va="bottom", fontweight="bold")
ax1.set_ylabel("Mean Composite Score")
ax1.set_ylim(0, 1)
ax1.set_xticks(x)
ax1.set_xticklabels(threshold_order, rotation=20, ha="right")

ax2 = ax1.twinx()
ax2.plot(x, thr_group["refine_calls_mean"], color="#D65F5F", marker="o", linewidth=2, label="Refine calls")
ax2.set_ylabel("Mean refine calls / image")
ax2.set_ylim(0, max(3.0, float(thr_group["refine_calls_mean"].max()) + 0.5))
ax1.set_title("Threshold Ablation: Composite Score and Refinement Calls")
plt.tight_layout()
save_figure(fig, "fig2_threshold_ablation")
plt.show()

table4 = thr_group.copy()
table4["mean_composite ± std"] = table4.apply(lambda r: f"{r['composite_mean']:.3f} ± {r['composite_std']:.3f}", axis=1)
table4["converged_pct"] = table4["converged_pct"] * 100.0
table4_display = table4[["mean_composite ± std", "refine_calls_mean", "converged_pct"]]
display(table4_display.round(3))
print("LaTeX source:")
print(table4_display.to_latex(float_format="%.3f"))

## 5. Figure 3: XAI Selector Ablation

In [ ]:
selector_order = ["always_gradcam", "rule_based", "oracle_mlp"]
selector_group = selector_ablation.groupby("selector_type").agg(
    accuracy=("correct", "mean"),
    composite=("composite", "mean"),
    composite_std=("composite", "std"),
).reindex(selector_order)
selector_colors = ["#8A8A8A", "#4878CF", "#C4AD66"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(selector_order, selector_group["accuracy"] * 100.0, color=selector_colors, edgecolor="black")
axes[0].set_ylabel("Selection Accuracy (%)")
axes[0].set_ylim(0, 100)
axes[0].set_title("Oracle Method Match")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(selector_order, selector_group["composite"], yerr=selector_group["composite_std"], capsize=4, color=selector_colors, edgecolor="black")
axes[1].set_ylabel("Mean Composite Score")
axes[1].set_ylim(0, 1)
axes[1].set_title("Composite Score")
axes[1].tick_params(axis="x", rotation=20)
axes[1].text(2, selector_group.loc["oracle_mlp", "composite"] + 0.05, "* Ours", ha="center", va="bottom", fontweight="bold")
fig.suptitle("XAI Selector Ablation: Method Selection Accuracy and Composite Score")
plt.tight_layout()
save_figure(fig, "fig3_selector_ablation")
plt.show()

display(selector_group.round(3))

## 6. Figure 4: Performance by Scene Complexity

In [ ]:
complexity_order = ["low", "medium", "high"]
complexity_colors = {"low": "#6ACC65", "medium": "#EEA236", "high": "#D65F5F"}
ours_df = main_results[main_results["method"] == "AED-XAI (Ours)"]
complexity_group = ours_df.groupby("scene_complexity")[["pg", "oa", "composite"]].mean().reindex(complexity_order)

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
x = np.arange(3)
width = 0.22
for idx, metric in enumerate(["pg", "oa", "composite"]):
    values = complexity_group[metric].values
    bars = ax.bar(x + (idx - 1) * width, values, width=width, label=metric.upper(), edgecolor="black")
    for bar, complexity in zip(bars, complexity_order):
        bar.set_facecolor(complexity_colors[complexity])
        bar.set_alpha(0.6 + idx * 0.12)

ax.set_xticks(x)
ax.set_xticklabels(complexity_order)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("AED-XAI Performance by Scene Complexity")
ax.legend(frameon=False)
plt.tight_layout()
save_figure(fig, "fig4_scene_complexity")
plt.show()

display(complexity_group.round(3))

## 7. Figure 5: Cross-Domain Generalization

In [ ]:
domain_order = ["COCO", "VOC", "VisDrone"]
heatmap_df = cross_domain.groupby(["domain", "method"])["composite"].mean().unstack("method").reindex(index=domain_order, columns=METHODS)

fig, ax = plt.subplots(1, 1, figsize=(10, 4.5))
im = ax.imshow(heatmap_df.values, cmap="YlOrRd", vmin=0.45, vmax=0.80)
ax.set_xticks(np.arange(len(METHODS)))
ax.set_xticklabels(METHODS, rotation=20, ha="right")
ax.set_yticks(np.arange(len(domain_order)))
ax.set_yticklabels(domain_order)
for i in range(heatmap_df.shape[0]):
    for j in range(heatmap_df.shape[1]):
        value = heatmap_df.iloc[i, j]
        ax.text(j, i, f"{value:.3f}", ha="center", va="center", color="black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Mean Composite Score")
ax.set_title("Cross-Domain Generalization (Composite Score)")
plt.tight_layout()
save_figure(fig, "fig5_cross_domain")
plt.show()

display(heatmap_df.round(3))

## 8. Figure 6: Qualitative Comparison (Paper Figure)

In [ ]:
XAI_CACHE = Path("../data/xai_comparison_cache.json")

def decode_saliency(encoded):
    payload = base64.b64decode(encoded["data"].encode("ascii"))
    raw = zlib.decompress(payload)
    return np.frombuffer(raw, dtype=np.float16).reshape(encoded["shape"]).astype(np.float32)

if not XAI_CACHE.exists():
    print("⚠ Run notebook 03 first to generate xai_comparison_cache.json")
else:
    with XAI_CACHE.open("r", encoding="utf-8") as handle:
        xai_records = json.load(handle)

    scored_records = []
    for record in xai_records:
        scores = {item["method"]: item["composite"] for item in record["methods"]}
        oracle = max(scores, key=scores.get)
        margin = sorted(scores.values(), reverse=True)[0] - sorted(scores.values(), reverse=True)[1]
        scored_records.append((record, oracle, margin, max(scores.values())))

    winners = sorted(scored_records, key=lambda item: item[2], reverse=True)[:2]
    failure = sorted(scored_records, key=lambda item: item[3])[0:1]
    selected = winners + failure

    fig = plt.figure(figsize=(16, 3.6 * len(selected)))
    gs = gridspec.GridSpec(len(selected), 5, figure=fig)
    for row_idx, (record, oracle, _, _) in enumerate(selected):
        image_path = Path("../data/coco/val2017") / record["filename"]
        if image_path.exists():
            image = plt.imread(image_path)
        else:
            image = np.ones((320, 480, 3), dtype=np.float32)
        det = record.get("detection", {})
        bbox = det.get("bbox", [0, 0, image.shape[1] - 1, image.shape[0] - 1])
        x1, y1, x2, y2 = bbox
        method_results = {item["method"]: item for item in record["methods"]}

        ax = fig.add_subplot(gs[row_idx, 0])
        ax.imshow(image)
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="lime", facecolor="none"))
        ax.set_title(f"{record['filename']}\noracle={oracle}", color="green")
        ax.axis("off")

        for col_idx, method in enumerate(["gradcam", "gcame", "dclose", "lime"], start=1):
            item = method_results[method]
            saliency = decode_saliency(item["saliency"])
            ax = fig.add_subplot(gs[row_idx, col_idx])
            ax.imshow(image)
            ax.imshow(saliency, cmap="jet", alpha=0.5, vmin=0, vmax=1)
            ax.set_title(f"{method}\nComp={item['composite']:.2f}")
            ax.axis("off")

    plt.tight_layout()
    save_figure(fig, "fig6_qualitative")
    plt.show()

## 9. Summary: All Figures Generated

In [ ]:
print("Files in figures directory:")
for path in sorted(FIGURES_DIR.glob("*")):
    size_kb = path.stat().st_size / 1024.0
    print(f"  {path.name:<36} {size_kb:8.1f} KB")

print("\nFigures for BMVC paper:")
for fig_name in [
    "fig1_composite_distribution",
    "fig2_threshold_ablation",
    "fig3_selector_ablation",
    "fig4_scene_complexity",
    "fig5_cross_domain",
    "fig6_qualitative",
]:
    path = FIGURES_DIR / f"{fig_name}.pdf"
    status = "✅" if path.exists() else "❌"
    print(f"  {status} {fig_name}.pdf")